# Phase 4: Time-Series Forecasting with XGBoost
## Industrial Demand Forecasting and Sales Analytics System

This notebook implements a supervised machine learning approach to forecast alloy demand using **XGBoost**. 

Steps:
1. Load the preprocessed and feature engineered data.
2. Encode categorical columns (`Product_Code`, `Region`, `Alloy_Type`).
3. Chronologically split into train and test sets.
4. Train a global XGBoost Regressor.
5. Predict on test set and evaluate.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")

### 1. Load Feature Engineered Data
We read in `data/output/engineered_features.csv` generated by the pipeline.

In [ ]:
features_path = os.path.join("..", "data", "output", "engineered_features.csv")
if not os.path.exists(features_path):
    print("Features file not found. Let's build them from raw data using src/run_pipeline.py first.")
else:
    df = pd.read_csv(features_path)
    df['Order_Date'] = pd.to_datetime(df['Order_Date'])
    print(f"Loaded dataset with {df.shape[0]} records and {df.shape[1]} columns.")

### 2. Supervised Learning Format & Split

In [ ]:
# Feature columns
features = [
    'Year', 'Month', 'Quarter', 'Week', 'Day_of_Week', 'Is_Weekend',
    'Month_Sin', 'Month_Cos', 'Lag_1', 'Lag_7', 'Lag_30',
    'Rolling_Mean_7', 'Rolling_Mean_30', 'Rolling_Std_7'
]

# One-hot encode categoricals
df_encoded = pd.get_dummies(df, columns=['Product_Code', 'Region', 'Alloy_Type'], drop_first=False)

# Get final feature list
dummy_cols = [c for c in df_encoded.columns if '_' in c and not c.startswith('Rolling_') and not c.startswith('Lag_') and not c.startswith('Month_') and not c.startswith('Day_') and not c.startswith('Is_') and not c.startswith('Order_Date') and not c == 'Quantity' and not c == 'Revenue' and not c == 'Customer_ID' and not c == 'Industry']
all_features = features + dummy_cols

# Split chronologically (last 90 days as test)
max_date = df_encoded['Order_Date'].max()
split_date = max_date - pd.Timedelta(days=90)

train_df = df_encoded[df_encoded['Order_Date'] <= split_date].dropna()
test_df = df_encoded[df_encoded['Order_Date'] > split_date].dropna()

X_train, y_train = train_df[all_features], train_df['Quantity']
X_test, y_test = test_df[all_features], test_df['Quantity']

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

### 3. Fit XGBoost Model

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=6,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("Model fitting complete.")

### 4. Feature Importance
Let's see which features are most important in determining alloy demand.

In [ ]:
importance = pd.DataFrame({
    'Feature': all_features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(10)

sns.barplot(data=importance, x='Importance', y='Feature', palette='crest')
plt.title('Top 10 Feature Importances - XGBoost')
plt.show()